# Caption video trên Kaggle: OmniShotCut → SpeedPredictor → InternVL3.5 + vLLM

Bật **Internet** và **GPU**. Notebook clone nhánh `vbs-dev`, lấy tối đa 10 video mỗi lô từ Drive, upload chỉ JSON caption (`video_id`, `start`, `end`, `caption`). Shot và JSON được giữ local; partial JSON checkpoint sau mỗi batch để resume. Không tạo file log/score/label.

Cell 2 cài `requirements-caption.txt`: vLLM 0.11.2 + Transformers 4.57.1 và bộ Torch/CUDA tương thích do vLLM yêu cầu, thay cho Torch có sẵn nếu cần. Inference chạy trong subprocess mới; notebook không import Torch/vLLM vào kernel. Các notebook khác dùng chung Python session cũng chịu thay đổi packages này.

Cell 3 chỉnh prompt, FPS, max frame, batch và VRAM. Mặc định **1.5 FPS, tối đa 16 frame, một tile 448×448/frame**, gom 8 shot/lần gửi; scheduler tối đa 4 shot đồng thời. Một thread chuẩn bị trước một batch trong khi GPU sinh caption. Giữ engine xuyên suốt lô video; không gc/empty_cache mỗi shot. Chunked prefill, prefix cache và CUDA graph/compilation bật; lần khởi động đầu có thể lâu. Có log tốc độ batch và tổng thời gian caption, không bao gồm model load/OMNI/speed. Chưa benchmark GPU thực tế ở máy phát triển.

OmniShotCut được giải phóng trước SpeedPredictor. SpeedPredictor chạy NVDEC + resize/normalize/inference CUDA trong process riêng; không import Decord/PyAV/OpenCV/torchvision/OmniShotCut. Dùng 16 frame/checkpoint, 7.5 FPS tham chiếu, 3 lượt thích nghi theo code gốc và FPS thực. Shot ngắn hơn 2.133s là unknown. Score >1.8: timelapse; <0.6: slow-motion; còn lại normal. Chỉ chèn nhãn vào prefix sau caption. Worker exit code/traceback được stream trực tiếp.

Console: OMNI/OMNI-SHOT → SPEED → CAPTION READY → CAPTION-BATCH → CAPTION-FINAL từng shot → CAPTION-PERF → JSON-CAPTION. Caption của một batch được in sau khi batch hoàn thành; READY của batch sau có thể xuất hiện trước FINAL của batch hiện tại do prefetch.

Đổi model/prompt/sampling có run ID và caption local riêng. Dùng folder Drive đích riêng để so sánh cùng video: mặc định JSON cùng video ID đã tồn tại trên Drive sẽ bị bỏ qua. Khi OOM, giảm MAX_NUM_SEQS/MAX_NUM_BATCHED_TOKENS hoặc bật ENFORCE_EAGER rồi chạy lại cell 3 + 5. Không tự giảm số frame hoặc retry engine đã crash; tiến độ batch đã lưu được tiếp tục. Đổi batch/VRAM không đổi run ID để resume được batch cũ; đổi frame/prompt có run ID mới. Giảm MAX_FRAMES xuống 12/8 là lựa chọn thử nghiệm chất lượng của bạn.

Frame caption được lọc trước resize bằng histogram RGB 8×8×8 trên toàn frame, gồm cả giá trị 255. Loại khi số bin có màu < MIN_COLOR_BINS (mặc định 5); đúng 5 bin vẫn giữ. Chỉ lọc tập frame đã sample, không tự lấy bù. Timestamp của frame giữ lại không thay đổi. Shot không còn frame hợp lệ được checkpoint là skipped và không có segment trong JSON cuối; toàn video bị lọc thì JSON là []. Log CAPTION-FILTER in từng frame bị loại và tổng kept/dropped. Filter có version/ngưỡng trong run ID để không dùng lại caption cũ.


In [ ]:
# 1. Cấu hình
import importlib.metadata, importlib.util, json, os, re, shutil, subprocess, sys, urllib.request, zipfile
from pathlib import Path
REPO_URL = 'https://github.com/iamtro-2006/event-retrieval-system.git'
REPO_BRANCH = 'vbs-dev'  # nhánh chứa pipeline caption
DATASET = 'V3C'
START_VIDEO_ID = 1
END_VIDEO_ID = 10
VIDEO_EXTENSION = '.mp4'
SOURCE_ROOT_FOLDER_ID = '19b7skL2b0PRJWWweWFh8qVNTLQDF6x9K'
DEST_ROOT_FOLDER_ID = '1I97-3-Y_V0-dfk-tPS4g7H8lpAHyeu2r'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
UPLOAD_BATCH_SIZE = 10
OVERWRITE_REMOTE = False

def video_id(number):
    return f'{number:05d}'

if DATASET != 'V3C' or not 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999:
    raise ValueError('Cần 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 99999 cho V3C')
if SOURCE_ROOT_FOLDER_ID == DEST_ROOT_FOLDER_ID:
    raise ValueError('Folder Drive nguồn và đích phải khác nhau')
if not RCLONE_CONFIG_SOURCE.is_file():
    raise FileNotFoundError(f'Chưa gắn rclone config: {RCLONE_CONFIG_SOURCE}')
WORK = Path('/kaggle/working/caption_pipeline')
WORK.mkdir(parents=True, exist_ok=True)
JOBS_ROOT = WORK / 'jobs'
JOBS_ROOT.mkdir(exist_ok=True)
RCLONE_CONFIG = WORK / 'rclone.conf'
shutil.copyfile(RCLONE_CONFIG_SOURCE, RCLONE_CONFIG)
RCLONE_CONFIG.chmod(0o600)
REPO_DIR = WORK / 'event-retrieval-system'
BACKEND_DIR = REPO_DIR / 'backend'
print('Workspace:', WORK)


In [ ]:
# 2. Clone/pull repo, cài dependencies chung + OmniShotCut và chuẩn bị rclone
def run(args, *, cwd=None, env=None):
    from collections import deque
    import queue
    import signal
    import threading
    import time
    args = [str(item) for item in args]
    print('+', ' '.join(args[:5]), '...' if len(args) > 5 else '', flush=True)
    tail = deque(maxlen=80)
    messages = queue.Queue()
    started = time.monotonic()
    last_output = started
    with subprocess.Popen(args, cwd=cwd, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors='replace',
                          bufsize=1) as process:
        def pump():
            try:
                for line in process.stdout:
                    messages.put(line)
            finally:
                messages.put(None)
        threading.Thread(target=pump, daemon=True).start()
        eof = False
        try:
            while not eof or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                except queue.Empty:
                    if time.monotonic() - last_output >= 30:
                        status = process.poll()
                        print(f'[RUN-HEARTBEAT] pid={process.pid} '
                              f'elapsed={time.monotonic()-started:.0f}s '
                              f'status={"running" if status is None else status}; '
                              'no new child output for 30s (not a progress estimate)', flush=True)
                        last_output = time.monotonic()
                    continue
                if line is None:
                    eof = True
                    continue
                print(line, end='', flush=True)
                tail.append(line)
                last_output = time.monotonic()
            exit_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print('[RUN-END] interrupted by user', flush=True)
            raise
    detail = ''
    if exit_code < 0:
        try:
            detail = f' signal={signal.Signals(-exit_code).name}'
        except ValueError:
            detail = f' signal={-exit_code}'
    print(f'[RUN-END] pid={process.pid} exit_code={exit_code}{detail} '
          f'elapsed={time.monotonic()-started:.1f}s', flush=True)
    if exit_code:
        raise RuntimeError(f'Command exited with code {exit_code}{detail}: {args[:5]}\n'
                           + ''.join(tail))
    return exit_code

if not REPO_DIR.exists():
    run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR])
else:
    branch = subprocess.check_output(['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
    if branch != REPO_BRANCH:
        raise RuntimeError(f'Checkout đang ở {branch}, cần {REPO_BRANCH}')
    run(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if not (BACKEND_DIR / 'src/caption_extraction/pipeline/extract_caption.py').is_file():
    raise RuntimeError('Nhánh GitHub chưa có caption extraction; hãy push code lên vbs-dev')
print('Caption commit:', commit)

# Bộ dependencies vLLM riêng; không import Torch trong kernel notebook.
def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None

requirements = BACKEND_DIR / 'requirements-caption.txt'
if (installed_version('vllm') != '0.11.2' or installed_version('transformers') != '4.57.1'
        or (installed_version('torch') or '').split('+')[0] != '2.9.0'
        or any(installed_version(name) is None for name in
               ('decord', 'PyNvVideoCodec', 'timm', 'accelerate', 'PyYAML', 'ffmpeg-python', 'einops', 'sentencepiece'))):
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed', '-r', requirements])
if importlib.util.find_spec('omnishotcut') is None:
    run([sys.executable, '-m', 'pip', 'install', '--upgrade-strategy', 'only-if-needed',
         'git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git'])
import yaml
run([sys.executable, '-u', '-c',
     'import torch, transformers, vllm; '
     'from vllm import LLM; '
     'assert torch.cuda.is_available(), "Enable GPU in Kaggle Settings"; '
     'print("GPU:", torch.cuda.get_device_name(0), "Torch:", torch.__version__, '
     '"CUDA:", torch.version.cuda, "vLLM:", vllm.__version__, "Transformers:", transformers.__version__, flush=True)'])

rclone = shutil.which('rclone')
if rclone is None and shutil.which('apt-get'):
    try:
        run(['apt-get', 'update', '-qq'])
        run(['apt-get', 'install', '-y', '-qq', 'rclone'])
        rclone = shutil.which('rclone')
    except subprocess.CalledProcessError:
        pass
if rclone is None:
    release = 'v1.75.2'
    archive = WORK / f'rclone-{release}-linux-amd64.zip'
    url = f'https://github.com/rclone/rclone/releases/download/{release}/{archive.name}'
    with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'kaggle-caption'}), timeout=120) as src, archive.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    with zipfile.ZipFile(archive) as zf:
        matches = [item for item in zf.infolist() if item.filename.endswith('/rclone') and not item.is_dir()]
        if len(matches) != 1:
            raise RuntimeError('Không tìm thấy binary rclone trong ZIP release')
        rclone = WORK / 'rclone'
        with zf.open(matches[0]) as src, rclone.open('wb') as dst:
            shutil.copyfileobj(src, dst)
    rclone.chmod(0o755)
run([rclone, 'version'])


In [ ]:
# 3. InternVL3.5 + vLLM: sửa tham số/prompt rồi chạy lại cell này + cell 5.
import hashlib
import math
import yaml

CAPTION_MODEL = 'OpenGVLab/InternVL3_5-2B'
FRAME_SAMPLE_FPS = 1.5
MAX_FRAMES = 16
MIN_COLOR_BINS = 5          # Loai frame co <5 bins mau; dung 5 bins van giu
CAPTION_BATCH_SIZE = 8      # Số shot gửi mỗi generate(); giới hạn RAM/prefetch
MAX_NUM_SEQS = 4            # Số shot scheduler xử lý đồng thời, ảnh hưởng VRAM
GPU_MEMORY_UTILIZATION = 0.88
MAX_MODEL_LEN = 6144
MAX_NUM_BATCHED_TOKENS = 4096
ENABLE_PREFIX_CACHING = True
ENFORCE_EAGER = False       # True bỏ CUDA graph/compile nếu cần giảm VRAM
MAX_NEW_TOKENS = 220
TIMELAPSE_THRESHOLD = 1.8    # score > ngưỡng này
SLOW_MOTION_THRESHOLD = 0.6  # score < ngưỡng này

CAPTION_PROMPT = (
    'Describe the following video segment in English. Begin with "The video segment". In 1-4 detailed sentences, focusing main visible subjects, their distinctive details and positions, the relevant surroundings at that moment. Describe what happens from the beginning to the end in reasonable chronological order, including meaningful changes, interactions between subjects, and camera movement when visible. Be specific about observable actions, scene, interaction and subject attribute details and identity rather than giving a broad summary. Include only what can be seen; omit uncertain or absent details without mentioning their absence. DO NOT GUESS any event, action or intent.'
)

vllm_config = {
    'gpu_memory_utilization': GPU_MEMORY_UTILIZATION,
    'max_model_len': MAX_MODEL_LEN,
    'max_num_seqs': MAX_NUM_SEQS,
    'max_num_batched_tokens': MAX_NUM_BATCHED_TOKENS,
    'enable_prefix_caching': ENABLE_PREFIX_CACHING,
    'enforce_eager': ENFORCE_EAGER,
}
# Shared validation imports no Torch/vLLM into the notebook kernel.
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
import importlib
import src.caption_extraction.pipeline.internvl as caption_runtime
caption_runtime = importlib.reload(caption_runtime)
validate_settings = caption_runtime.validate_settings
vllm_config = validate_settings(CAPTION_MODEL, MAX_FRAMES, CAPTION_BATCH_SIZE, vllm_config)
if type(MIN_COLOR_BINS) is not int or not 1 <= MIN_COLOR_BINS <= 512:
    raise ValueError('MIN_COLOR_BINS must be an integer in [1, 512]')
if not math.isfinite(FRAME_SAMPLE_FPS) or FRAME_SAMPLE_FPS <= 0:
    raise ValueError('FRAME_SAMPLE_FPS must be positive and finite')
if type(MAX_NEW_TOKENS) is not int or MAX_NEW_TOKENS <= 0 or not CAPTION_PROMPT.strip():
    raise ValueError('MAX_NEW_TOKENS must be positive; prompt must not be empty')
if not (math.isfinite(SLOW_MOTION_THRESHOLD) and math.isfinite(TIMELAPSE_THRESHOLD)
        and 0 < SLOW_MOTION_THRESHOLD < TIMELAPSE_THRESHOLD):
    raise ValueError('Need 0 < SLOW_MOTION_THRESHOLD < TIMELAPSE_THRESHOLD')

config_path = BACKEND_DIR / 'configs/caption_extraction.yaml'
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
extraction = cfg['extraction']
extraction.pop('caption_profiles', None)
extraction.pop('internvl', None)
extraction.update(caption_model=CAPTION_MODEL, caption_max_frames=MAX_FRAMES,
                  caption_batch_size=CAPTION_BATCH_SIZE, caption_min_color_bins=MIN_COLOR_BINS, vllm=vllm_config,
                  prompt=CAPTION_PROMPT.strip(), frame_sample_fps=float(FRAME_SAMPLE_FPS),
                  max_new_tokens=MAX_NEW_TOKENS)
extraction['speed_predictor']['timelapse_threshold'] = float(TIMELAPSE_THRESHOLD)
extraction['speed_predictor']['slow_motion_threshold'] = float(SLOW_MOTION_THRESHOLD)
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
# Runtime batch/VRAM tuning keeps the same output/checkpoint directory.
run_config = {key: value for key, value in extraction.items() if key not in {'caption_batch_size', 'vllm'}}
run_config['caption_backend'] = 'vllm-internvl-v1'
run_config['frame_filter_version'] = caption_runtime.CAPTION_FRAME_FILTER_VERSION
CAPTION_RUN_ID = hashlib.sha256(yaml.safe_dump(run_config, sort_keys=True).encode()).hexdigest()[:10]
STAGE_DIR = WORK / 'pending_json_2b' / CAPTION_RUN_ID
STAGE_DIR.mkdir(parents=True, exist_ok=True)
print('Caption model:', CAPTION_MODEL, '| backend: vLLM | run:', CAPTION_RUN_ID, flush=True)
print('Sampling FPS:', FRAME_SAMPLE_FPS, '| max frames:', MAX_FRAMES, '| tile: model-native (448)', flush=True)
print('Frame filter: full RGB histogram 8x8x8 before resize, drop bins <', MIN_COLOR_BINS, flush=True)
print('Batch size:', CAPTION_BATCH_SIZE, '| scheduler/VRAM:', vllm_config, flush=True)
print('SpeedPredictor:', extraction['speed_predictor'], flush=True)
print('Caption prompt:', CAPTION_PROMPT, flush=True)
print('Live logs: OMNI -> SPEED -> CAPTION-BATCH -> CAPTION-FINAL -> CAPTION-PERF -> JSON-CAPTION', flush=True)
print('Local captions:', WORK / 'captions_2b' / CAPTION_RUN_ID / DATASET, flush=True)
print('Chạy cell 5 để xử lý; Drive vẫn bỏ qua ID đã có JSON trừ khi OVERWRITE_REMOTE=True.', flush=True)


In [ ]:
# 4. Liệt kê video nguồn và bỏ qua caption JSON đã upload
def remote_args(folder_id):
    return [rclone, '--config', RCLONE_CONFIG, '--drive-root-folder-id', folder_id]

source_names = set(subprocess.check_output(
    [*map(str, remote_args(SOURCE_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
selected = [video_id(number) for number in range(START_VIDEO_ID, END_VIDEO_ID + 1)]
missing = [vid for vid in selected if f'{vid}{VIDEO_EXTENSION}' not in source_names]
if missing:
    raise FileNotFoundError(f'Thiếu video trong Drive nguồn: {missing}')
dest_names = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
pending = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in dest_names]
print(f'Đã chọn {len(selected)} video; cần xử lý {len(pending)}:', pending)


In [ ]:
# 5. Từng lô: tải video cần xử lý, tạo shot + caption, upload JSON rồi dọn video tạm
import math
cfg = yaml.safe_load((BACKEND_DIR / 'configs/caption_extraction.yaml').read_text(encoding='utf-8'))
if cfg['extraction']['caption_model'] != CAPTION_MODEL:
    raise RuntimeError('Caption config trên nhánh này chưa đúng model')
CAPTION_ENV = os.environ.copy()
CAPTION_ENV['PYTHONPATH'] = str(BACKEND_DIR)
CAPTION_ENV['PYTHONUNBUFFERED'] = '1'
CAPTION_ENV['PYTHONFAULTHANDLER'] = '1'
CAPTION_ENV['HF_HOME'] = str(WORK / 'hf_cache')
CAPTION_ENV['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
CAPTION_ENV['TOKENIZERS_PARALLELISM'] = 'false'
CAPTION_ENV['OMP_NUM_THREADS'] = '2'
SCENES_DIR = WORK / 'scenes' / DATASET
CAPTIONS_DIR = WORK / 'captions_2b' / CAPTION_RUN_ID / DATASET
SCENES_DIR.mkdir(parents=True, exist_ok=True)
CAPTIONS_DIR.mkdir(parents=True, exist_ok=True)

def validate_json(path, vid):
    data = json.loads(path.read_text(encoding='utf-8'))
    if not isinstance(data, list):
        raise ValueError(f'{path}: cần danh sách caption')
    if not data:
        print(f'{vid}: no caption segments (all sampled frames were filtered); valid empty JSON.', flush=True)
    previous_start = -1.0
    for item in data:
        if (not isinstance(item, dict) or set(item) != {'video_id', 'start', 'end', 'caption'} or
                item['video_id'] != vid or not isinstance(item['caption'], str) or
                not item['caption'].strip() or
                not isinstance(item['start'], (int, float)) or
                not isinstance(item['end'], (int, float)) or
                not math.isfinite(item['start']) or not math.isfinite(item['end']) or
                not 0 <= item['start'] < item['end'] or item['start'] < previous_start):
            raise ValueError(f'{path}: segment không đúng schema')
        previous_start = item['start']
    return len(data)

# Restore staged JSON after an interrupted upload; completed files remain in CAPTIONS_DIR.
for staged in STAGE_DIR.glob('*.json'):
    local = CAPTIONS_DIR / staged.name
    if not local.is_file() or staged.read_bytes() != local.read_bytes():
        raise RuntimeError(f'JSON staged không khớp bản local: {staged}')
    staged.unlink()

# Recheck remote on cell rerun, so completed batches are not uploaded again.
remote_done = set(subprocess.check_output(
    [*map(str, remote_args(DEST_ROOT_FOLDER_ID)), 'lsf', f'{RCLONE_REMOTE}:', '--files-only'],
    text=True).splitlines())
remaining = [vid for vid in selected if OVERWRITE_REMOTE or f'{vid}.json' not in remote_done]
completed = {}
for batch_number, start in enumerate(range(0, len(remaining), UPLOAD_BATCH_SIZE)):
    batch_ids = remaining[start:start + UPLOAD_BATCH_SIZE]
    batch_work = JOBS_ROOT / f'batch_{batch_number:04d}'
    video_dir = batch_work / 'videos' / DATASET
    video_dir.mkdir(parents=True, exist_ok=True)
    need_processing = [vid for vid in batch_ids if not (CAPTIONS_DIR / f'{vid}.json').is_file()]
    for vid in need_processing:
        path = video_dir / f'{vid}{VIDEO_EXTENSION}'
        if not path.is_file() or path.stat().st_size == 0:
            run([*remote_args(SOURCE_ROOT_FOLDER_ID), 'copyto',
                 f'{RCLONE_REMOTE}:{path.name}', path])
        else:
            print('Dùng lại video đã tải:', path.name)
        if not path.is_file() or path.stat().st_size == 0:
            raise RuntimeError(f'Video tải về rỗng: {path.name}')
    if need_processing:
        cmd = [sys.executable, '-u', '-m', 'scripts.caption.extract.run', '--dataset', DATASET.lower(),
               '--input-dir', video_dir, '--scenes-dir', SCENES_DIR, '--output-dir', CAPTIONS_DIR]
        for vid in need_processing:
            cmd += ['--video-id', vid]
        run(cmd, cwd=BACKEND_DIR, env=CAPTION_ENV)
    for vid in batch_ids:
        result = CAPTIONS_DIR / f'{vid}.json'
        if not result.is_file():
            raise FileNotFoundError(f'Không có caption JSON: {result}')
        completed[vid] = validate_json(result, vid)
        shutil.copy2(result, STAGE_DIR / result.name)
        print(f'Caption xong {vid}: {completed[vid]} segment')
    for path in video_dir.glob(f'*{VIDEO_EXTENSION}'):
        path.unlink()  # giữ shot/JSON trong WORK, giải phóng video trước khi upload
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'copy', STAGE_DIR, f'{RCLONE_REMOTE}:',
         '--transfers', str(min(UPLOAD_BATCH_SIZE, len(batch_ids))), '--checkers', '10'])
    run([*remote_args(DEST_ROOT_FOLDER_ID), 'check', STAGE_DIR, f'{RCLONE_REMOTE}:', '--one-way'])
    for path in STAGE_DIR.glob('*.json'):
        path.unlink()
    shutil.rmtree(batch_work)
    print(f'Đã upload {len(batch_ids)} caption JSON; giữ shot/JSON local, dọn video tạm')
print('Hoàn tất:', completed, '| commit:', commit)
